# qkdpp: post-processing an experimental QKD dataset

Runs the full classical post-processing pipeline -- **sifting, parameter estimation, error correction, verification, and privacy amplification** -- on a raw dataset from a lab experiment, using only the [`qkdpp`](../README.md) package.

Place your raw CSV in the same folder as this notebook (or adjust `CSV_PATH` below).

The sifting rule in this notebook is specific to the column schema of the dataset it was 
written against (a phase-encoded, Sagnac-style protocol). If your dataset uses a different 
schema, adjust the rule in section 1 -- the rest of the pipeline is protocol-agnostic and 
does not need to change.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import qkdpp

## 1. Load raw data and apply sifting

Sifting rule used here, validated against a known-good sifted pair (100% bit-for-bit match):

- `base_match == 1` -- Alice and Bob chose the same measurement basis
- `detectada == 1` -- a detection event occurred
- `detector != 'ambos'` -- excludes ambiguous double-click events

Alice's sifted bit is `bitA`. Bob's sifted bit is `bitA XOR error` -- the `error` column is, by definition, the per-round disagreement between Alice's key bit and Bob's detection outcome.

In [ ]:
CSV_PATH = Path("rondas_completas.csv")

raw = pd.read_csv(CSV_PATH)
print(f"raw rounds: {len(raw)}")

required_cols = {"round_id", "base_match", "detectada", "detector", "bitA", "error"}
missing = required_cols - set(raw.columns)
assert not missing, f"expected columns missing: {missing}"

mask = (raw.base_match == 1) & (raw.detectada == 1) & (raw.detector != "ambos")
sifted = raw[mask].sort_values("round_id")
assert sifted["error"].notna().all(), "sifting selected rows with no error flag -- wrong rule for this dataset"

alice_bits = sifted["bitA"].astype(int).to_numpy().astype(np.uint8)
bob_bits = (sifted["bitA"].astype(int) ^ sifted["error"].astype(int)).to_numpy().astype(np.uint8)

n_sifted = len(alice_bits)
qber_true = float(np.mean(alice_bits != bob_bits))
print(f"sifted bits: {n_sifted}")
print(f"true QBER (diagnostic only -- not used by the pipeline, which only sees a public sample): {qber_true:.4f}")

## 2. Run the qkdpp pipeline

`sifting.estimate_qber` (sacrifice a public sample, measure the error rate) -> `cascade.reconcile` (error correction, every revealed bit metered) -> `extract.verify` (Toeplitz-hash check) -> `extract.amplify` (privacy amplification), all wired together by `qkdpp.run(...)`.

By default this uses the *raw* public-sample error-rate estimate to size privacy amplification -- not a statistically conservative confidence bound. That's a deliberate design choice for this package (see the README's "Design philosophy" section); pass a different `e_ph` if you need a defensible security margin instead of a point estimate.

In [ ]:
result = qkdpp.run(alice_bits, bob_bits, pe_fraction=0.1, n_passes=10, seed=1)
print(result.summary())
print("leakage breakdown:", result.channel.breakdown)

## 3. Save the final key

In [ ]:
out_path = Path("final_key_qkdpp.txt")
qkdpp.io.save_bits(out_path, result.key)
print(f"final key ({result.final_len} bits) saved to {out_path}")

## 4. Efficiency benchmark (optional)

Since we have both Alice's and Bob's full sifted strings here, we can compute the true QBER directly and use it to ask: how close did the pipeline get to the Shannon-optimal final key length? This is a diagnostic for pipeline efficiency, not a security bound -- a real deployment never gets to use the true QBER for free.

In [ ]:
def h2(p):
    if p <= 0:
        return 0.0
    return -p * np.log2(p) - (1 - p) * np.log2(1 - p)

pa_cost = 2 * np.log2(1 / (2 * 1e-10))
leak_ec_ideal = n_sifted * h2(qber_true)
final_ideal = int(np.floor(n_sifted * (1 - h2(qber_true)) - leak_ec_ideal - pa_cost))

print(f"Shannon-optimal final length (true QBER, f_ec = 1.0): {final_ideal} bits")
print(f"qkdpp final length:                                  {result.final_len} bits")
print(f"efficiency relative to the Shannon bound:              {100 * result.final_len / final_ideal:.1f}%")